In [1]:
import pandas as pd
import numpy as np
import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import xgboost as xgb
from imblearn.over_sampling import SMOTE

In [2]:
df_sample = pd.read_csv('mid_project/sample_submission.csv')
df_test = pd.read_csv('mid_project/test_copy.csv')
df_train = pd.read_csv('mid_project/train_df.csv')

In [3]:
df_sample.shape, df_test.shape, df_train.shape

((48108, 2), (48108, 8), (50512, 9))

In [4]:
X = df_train.drop('target', axis=1)
y = df_train['target']
X.shape, y.shape

((50512, 8), (50512,))

In [6]:
smote = SMOTE()
X_resampled, y_resampled = smote.fit_resample(X, y)

In [7]:
y_resampled.value_counts()

target
0    47014
1    47014
Name: count, dtype: int64

In [8]:
X_resampled.shape, y_resampled.shape

((94028, 8), (94028,))

In [9]:
def objective(trial):

    params = {

        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1, 100),

        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 6),
        'eval_metric': 'auc',
        'random_state': 4,
        'verbosity': 0,
        'nthread': -1
    }

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=5)
    scores = []

    for fold, (train_index, val_index) in enumerate(skf.split(X_resampled, y_resampled)):
        X_train, X_val = X_resampled.iloc[train_index], X_resampled.iloc[val_index]
        y_train, y_val = y_resampled.iloc[train_index], y_resampled.iloc[val_index]

        model = xgb.XGBClassifier(**params)
        model.fit(X_train, y_train)
        y_pred = model.predict_proba(X_val)[:, 1]
        roc_auc = roc_auc_score(y_val, y_pred)
        scores.append(roc_auc)

    return np.mean(scores)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=40)

[I 2026-02-10 13:54:30,591] A new study created in memory with name: no-name-288b9721-fe5c-47ab-b0ae-26d4df818453
[I 2026-02-10 13:54:33,889] Trial 0 finished with value: 0.9085799770246895 and parameters: {'min_child_weight': 6, 'subsample': 0.5717783398335294, 'colsample_bytree': 0.8423933670438485, 'reg_alpha': 0.6903993394351999, 'reg_lambda': 66.21852131135421, 'n_estimators': 817, 'learning_rate': 0.05953123970378463, 'max_depth': 5}. Best is trial 0 with value: 0.9085799770246895.
[I 2026-02-10 13:54:34,934] Trial 1 finished with value: 0.8845009981417918 and parameters: {'min_child_weight': 10, 'subsample': 0.527243219933655, 'colsample_bytree': 0.5037459061401013, 'reg_alpha': 1.7648797940117854e-07, 'reg_lambda': 15.703107060990046, 'n_estimators': 256, 'learning_rate': 0.023866764673016485, 'max_depth': 5}. Best is trial 0 with value: 0.9085799770246895.
[I 2026-02-10 13:54:37,179] Trial 2 finished with value: 0.9201372291143972 and parameters: {'min_child_weight': 2, 'subsa

In [10]:
best_params = study.best_params
best_params

{'min_child_weight': 3,
 'subsample': 0.7792782344743387,
 'colsample_bytree': 0.7777371893151498,
 'reg_alpha': 0.8828911603485132,
 'reg_lambda': 31.57910904987173,
 'n_estimators': 887,
 'learning_rate': 0.08489517169340823,
 'max_depth': 6}

In [11]:
print(f'ROC AUC score :{study.best_value:.4f}')

ROC AUC score :0.9254


In [12]:
smote_xgb_model = xgb.XGBClassifier(**best_params, random_state=5)

In [13]:
smote_xgb_model.fit(X, y)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.7777371893151498
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [14]:
y_pred_xgb = smote_xgb_model.predict_proba(df_test)

In [15]:
df_sample['Predicted'] = y_pred_xgb
df_sample.head()

,Id,Predicted
0,1,0.875826
1,2,0.459409
2,3,0.984085
3,4,0.991707
4,5,0.640518


In [ ]:
df_sample.to_csv('submission.csv', index=False)

In [16]:
# import joblib
# joblib.dump(smote_xgb_model, 'smote_xgb.joblib')

['smote_xgb.joblib']